In [9]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
from models.wave2vec2.audio import to_tensors

In [10]:
class AudioSegmentDataset(Dataset):
    def __init__(self, label_dir, input_dir, num_samples=None):
        self.label_dir = label_dir
        self.input_dir = input_dir

        # Get all CSV files
        self.csv_files = os.listdir(label_dir)
        if num_samples:
            self.csv_files = self.csv_files[:num_samples]

        # Pre-load all data
        self.data = self._load_data()

    def _load_data(self):
        loaded_data = []
        for file in self.csv_files:
            file_path = os.path.join(self.label_dir, file)
            file_name = os.path.basename(file).split(".")[0]

            # Load labels
            df = pd.read_csv(file_path)

            # Load audio features
            audio_file = os.path.join(self.input_dir, f"{file_name}.mp3")
            feature_vectors = to_tensors(audio_file, segment_length_ms=50)

            # Store features and labels
            loaded_data.append({
                'features': feature_vectors,
                'breaks': torch.tensor(df['break'].values, dtype=torch.float32),
                'start_times': df['start'].values,
                'end_times': df['end'].values
            })
        return loaded_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# Usage example:
def create_dataloaders(label_dir, input_dir, num_samples=None, batch_size=1):
    dataset = AudioSegmentDataset(label_dir, input_dir, num_samples)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    return dataloader

In [11]:
data_dir = '../../data'
label_dir = f'{data_dir}/clean/segment_breaks'
input_dir = f'{data_dir}/clean/audio/vocals'

dataloader = create_dataloaders(
    label_dir=label_dir,
    input_dir=input_dir,
    num_samples=10,
    batch_size=1
)

/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/transformers/configuration_utils.py:306: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/transformers/configuration_utils.py:306: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/transformers/configuration_utils.py:306: 

In [12]:
# print information about all the samples
for sample in dataloader:
    print(f"Sample features shape: {sample['features'].shape}")
    print(f"Sample breaks shape: {sample['breaks'].shape}")
    print(f"Sample breaks: {sample['breaks'].shape}")

Sample features shape: torch.Size([1, 4414, 768])
Sample breaks shape: torch.Size([1, 4416])
Sample breaks: torch.Size([1, 4416])
Sample features shape: torch.Size([1, 5779, 768])
Sample breaks shape: torch.Size([1, 5780])
Sample breaks: torch.Size([1, 5780])
Sample features shape: torch.Size([1, 4924, 768])
Sample breaks shape: torch.Size([1, 4925])
Sample breaks: torch.Size([1, 4925])
Sample features shape: torch.Size([1, 4900, 768])
Sample breaks shape: torch.Size([1, 4901])
Sample breaks: torch.Size([1, 4901])
Sample features shape: torch.Size([1, 3574, 768])
Sample breaks shape: torch.Size([1, 3575])
Sample breaks: torch.Size([1, 3575])
Sample features shape: torch.Size([1, 3977, 768])
Sample breaks shape: torch.Size([1, 3978])
Sample breaks: torch.Size([1, 3978])
Sample features shape: torch.Size([1, 5662, 768])
Sample breaks shape: torch.Size([1, 5663])
Sample breaks: torch.Size([1, 5663])
Sample features shape: torch.Size([1, 4324, 768])
Sample breaks shape: torch.Size([1, 4325

In [29]:
class RNNModel(nn.Module):
    def __init__(self, input_size=768, hidden_size=256, num_layers=1, dropout=0, bidirectional=False):
        super(RNNModel, self).__init__()
        self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout, bidirectional=bidirectional)
        self.fc = nn.Linear(hidden_size * (2 if bidirectional else 1), 1)

    def forward(self, x):
        h0 = torch.zeros(self.rnn.num_layers * (2 if self.rnn.bidirectional else 1), x.size(0), self.rnn.hidden_size).to(x.device)
        c0 = torch.zeros(self.rnn.num_layers * (2 if self.rnn.bidirectional else 1), x.size(0), self.rnn.hidden_size).to(x.device)

        out, _ = self.rnn(x, (h0, c0))
        out = self.fc(out)
        out = torch.sigmoid(out)
        return out

In [35]:
import torch
import torch.nn.functional as F

class CustomLoss(torch.nn.Module):
    def __init__(self, alpha=0.5, beta=1.0):
        """
        Custom loss function combining BCE and proximity loss
        
        Args:
        - alpha (float): Weight for BCE loss (0 <= alpha <= 1)
        - beta (float): Proximity loss sensitivity parameter
        """
        super(CustomLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, y_pred, y_true):
        """
        Compute custom loss
        
        Args:
        - y_pred (torch.Tensor): Predicted break probabilities (batch x time_steps)
        - y_true (torch.Tensor): Ground truth break labels (batch x time_steps)
        
        Returns:
        - torch.Tensor: Computed loss
        """
        # Ensure inputs are on the same device and have correct shape
        y_pred = y_pred.squeeze()
        y_true = y_true.squeeze()

        # BCE Loss
        f_bce = F.binary_cross_entropy(y_pred, y_true, reduction='sum')

        # Proximity Loss
        f_proximity = self._compute_proximity_loss(y_pred, y_true)

        # Total Loss
        f_total = self.alpha * f_bce + (1 - self.alpha) * f_proximity

        return f_total

    def _compute_proximity_loss(self, y_pred, y_true):
        """
        Compute proximity loss
        
        Args:
        - y_pred (torch.Tensor): Predicted break probabilities
        - y_true (torch.Tensor): Ground truth break labels
        
        Returns:
        - torch.Tensor: Proximity loss
        """
        # Find indices of predicted and true breaks
        pred_breaks = torch.where(y_pred > 0.5)[0]
        true_breaks = torch.where(y_true > 0.5)[0]

        # If no breaks predicted or no true breaks, return 0
        if len(pred_breaks) == 0 or len(true_breaks) == 0:
            return torch.tensor(0.0, device=y_pred.device)

        # Compute proximity loss
        proximity_loss = 0.0
        for t in pred_breaks:
            # Find closest true break
            distances = torch.abs(t - true_breaks)
            min_distance = torch.min(distances)

            # Compute proximity term
            proximity_term = torch.exp(self.beta * min_distance) - 1
            proximity_loss += proximity_term

        return proximity_loss

def train_model(model, dataloader, criterion, optimizer, epochs):
    for epoch in range(epochs):
        total_loss = 0
        for batch_idx, batch in enumerate(dataloader):
            features = batch['features']
            targets = batch['breaks']

            # Truncate the target sequence to match the length of the input sequence
            targets = targets[:, :features.shape[1]]

            optimizer.zero_grad()
            outputs = model(features)

            # Compute loss using custom loss function
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")


In [36]:
print(f"Initializing the model")
model = RNNModel()
# criterion = torch.nn.BCELoss()
criterion = CustomLoss(alpha=0.5, beta=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Initializing the model


In [37]:
train_model(model, dataloader, criterion, optimizer, epochs=10)

Epoch 1/10, Loss: inf
Epoch 2/10, Loss: 832.0464
Epoch 3/10, Loss: 803.5772
Epoch 4/10, Loss: 782.7787
Epoch 5/10, Loss: 769.0978
Epoch 6/10, Loss: 760.3058
Epoch 7/10, Loss: 750.3471
Epoch 8/10, Loss: 739.8418
Epoch 9/10, Loss: 734.2487
Epoch 10/10, Loss: 730.8592
